# **Data Cleaning Notebook**

## Objectives

* Inspect the collected Heritage Housing datasets for data quality issues.
* Check duplicated rows.
* Investigate missing values.
* Apply cleaning decisions that are supported by the dataset metadata.
* Split the main housing dataset into train and test sets.
* Save cleaned datasets for feature engineering and modelling.

## Inputs

* `outputs/datasets/collection/house_prices_records.csv`
* `outputs/datasets/collection/inherited_houses.csv`

## Outputs

* `outputs/datasets/cleaned/TrainSetCleaned.csv`
* `outputs/datasets/cleaned/TestSetCleaned.csv`
* `outputs/datasets/cleaned/InheritedHousesCleaned.csv`

## Additional Comments

* This notebook focuses only on data cleaning.
* Feature engineering, encoding, transformations, and model preparation will be handled in later notebooks.
* Missing values are handled according to their meaning where this can be inferred from the dataset metadata.

---

# Set Working Directory

Ensure the working directory is set to the project root directory.

In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "jupyter_notebooks":
    os.chdir(current_dir.parent)
    print("You set a new current directory")
else:
    print("Current directory is already the project root or another non-notebook directory")

Path.cwd()

---

# Import Packages

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

---

# Load Datasets

In [ ]:
house_prices_df = pd.read_csv("outputs/datasets/collection/house_prices_records.csv")
inherited_houses_df = pd.read_csv("outputs/datasets/collection/inherited_houses.csv")

In [ ]:
house_prices_df.head()

In [ ]:
inherited_houses_df.head()

In [ ]:
house_prices_df.shape, inherited_houses_df.shape

# Check Dataset Columns

`inherited_houses_df` should contain only the feature columns, not the target, `SalePrice`.

In [ ]:
house_prices_df.columns.to_list()

In [ ]:
inherited_houses_df.columns.to_list()

In [ ]:
target = "SalePrice"
feature_columns = [col for col in house_prices_df.columns if col != target]

missing_from_inherited = set(feature_columns) - set(inherited_houses_df.columns)
extra_in_inherited = set(inherited_houses_df.columns) - set(feature_columns)

print("Missing from inherited houses:", missing_from_inherited)
print("Extra in inherited houses:", extra_in_inherited)

Since both calculations result in empty sets, the inherited houses dataset contains all the same feature columns of the house prices dataset, except the target variable, `SalePrice`.  

# Check Duplicated Rows

The following cells check whether the datasets contain duplicated rows.

In [ ]:
house_prices_df.duplicated().sum()

In [ ]:
inherited_houses_df.duplicated().sum()

There seem to be no duplicates, but this can be ensured too:

In [ ]:
house_prices_df = house_prices_df.drop_duplicates()
inherited_houses_df = inherited_houses_df.drop_duplicates()

house_prices_df.shape, inherited_houses_df.shape

# Check Missing Values

In [ ]:
def missing_values_table(df):
    """
    Create a table showing columns with missing values.
    """
    missing_count = df.isna().sum()
    missing_percentage = (missing_count / len(df)) * 100

    missing_df = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percentage": missing_percentage.round(2)
    })

    missing_df = missing_df[missing_df["missing_count"] > 0]
    return missing_df.sort_values(by="missing_count", ascending=False)

In [ ]:
missing_values_table(house_prices_df)

In [ ]:
missing_values_table(inherited_houses_df)

The dataset contains missing values in some features. The cleaning approach depends on the meaning of each missing value.

Some missing values represent the absence of a feature, such as no basement or no garage. Other missing values represent unknown numerical information and need a different strategy.

# Investigate Missing Values

## Missing-Value Strategy

The missing-value strategy separates missing values into three groups:

1. **Structural absence**: the missing value appears to represent the absence of a feature, such as no garage or no basement.
2. **Unknown category**: the feature appears to exist, but its category is missing.
3. **Unknown numerical value**: the numerical value is missing and cannot be inferred directly from another column.

This distinction matters because replacing every missing categorical value with `"None"` would be misleading. In this dataset, `"None"` has a specific meaning: no garage or no basement. Therefore, it should only be used where other variables support that interpretation.

In [ ]:
garage_finish_missing = house_prices_df[house_prices_df["GarageFinish"].isna()]

garage_finish_missing["GarageArea"].eq(0).value_counts()

In [ ]:
garage_finish_missing.groupby(garage_finish_missing["GarageArea"].eq(0))["GarageArea"].describe()

In [ ]:
bsmt_exposure_missing = house_prices_df[house_prices_df["BsmtExposure"].isna()]

bsmt_exposure_missing["TotalBsmtSF"].eq(0).value_counts()

In [ ]:
bsmt_fin_type_missing = house_prices_df[house_prices_df["BsmtFinType1"].isna()]

bsmt_fin_type_missing["TotalBsmtSF"].eq(0).value_counts()

The missing-value checks show that some missing values are structural, while others are unknown categories.

For `GarageFinish`, missing values where `GarageArea == 0` can reasonably be interpreted as `"None"`, meaning no garage. However, missing values where `GarageArea > 0` cannot be treated as `"None"` because the house appears to have a garage.

The same principle is applied to basement variables. Missing basement category values are only treated as `"None"` where `TotalBsmtSF == 0`.

# Split Train and Test Sets

The dataset is split before imputation. Any imputation values such as medians or modes are calculated from the training set only, then applied to the test set and inherited houses dataset.

This avoids using information from the test set when preparing the model data.

In [ ]:
TrainSet, TestSet = train_test_split(
    house_prices_df,
    test_size=0.2,
    random_state=0
)

TrainSet.shape, TestSet.shape

# Apply Structural Missing-Value Cleaning

Some missing values can be handled using logical relationships between columns.

For example, if `GarageArea` is `0`, then a missing `GarageFinish` value can reasonably be interpreted as `"None"`, meaning no garage. However, if `GarageArea` is greater than `0`, the house appears to have a garage, so the missing value should not be replaced with `"None"`.

The same logic applies to basement variables. Missing basement category values are replaced with `"None"` only where `TotalBsmtSF` is `0`.

This step avoids incorrectly labelling an existing garage or basement as absent.

In [ ]:
def apply_structural_missing_values(df):
    """
    Replace missing values only when other columns show that the feature is absent.
    """
    df = df.copy()

    # No garage:
    # If GarageArea is 0, then a missing GarageFinish can reasonably mean "No Garage".
    if {"GarageArea", "GarageFinish"}.issubset(df.columns):
        no_garage_mask = df["GarageArea"].eq(0)
        df.loc[no_garage_mask & df["GarageFinish"].isna(), "GarageFinish"] = "None"

    # If GarageArea is 0, a missing GarageYrBlt can reasonably be represented as 0.
    if {"GarageArea", "GarageYrBlt"}.issubset(df.columns):
        no_garage_mask = df["GarageArea"].eq(0)
        df.loc[no_garage_mask & df["GarageYrBlt"].isna(), "GarageYrBlt"] = 0

    # No basement:
    # If total basement area is 0, then missing basement category values can reasonably mean "No Basement".
    if {"TotalBsmtSF", "BsmtExposure"}.issubset(df.columns):
        no_basement_mask = df["TotalBsmtSF"].eq(0)
        df.loc[no_basement_mask & df["BsmtExposure"].isna(), "BsmtExposure"] = "None"

    if {"TotalBsmtSF", "BsmtFinType1"}.issubset(df.columns):
        no_basement_mask = df["TotalBsmtSF"].eq(0)
        df.loc[no_basement_mask & df["BsmtFinType1"].isna(), "BsmtFinType1"] = "None"

    # Masonry veneer:
    # Missing MasVnrArea is rare, and 0 is a common valid value meaning no masonry veneer area.
    if "MasVnrArea" in df.columns:
        df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

    return df

In [ ]:
TrainSet = apply_structural_missing_values(TrainSet)
TestSet = apply_structural_missing_values(TestSet)
inherited_houses_df = apply_structural_missing_values(inherited_houses_df)

# Apply Logical Category Cleaning

In [ ]:
def apply_logical_category_values(df):
    """
    Apply additional logical category values where a category can be inferred.
    """
    df = df.copy()

    # If the basement exists but BsmtFinSF1 is 0, then the type 1 finished area
    # is effectively unfinished.
    if {"TotalBsmtSF", "BsmtFinSF1", "BsmtFinType1"}.issubset(df.columns):
        unfinished_basement_mask = (
            df["TotalBsmtSF"].gt(0)
            & df["BsmtFinSF1"].eq(0)
            & df["BsmtFinType1"].isna()
        )
        df.loc[unfinished_basement_mask, "BsmtFinType1"] = "Unf"

    return df

In [ ]:
TrainSet = apply_logical_category_values(TrainSet)
TestSet = apply_logical_category_values(TestSet)
inherited_houses_df = apply_logical_category_values(inherited_houses_df)

# Impute Remaining Categorical Missing Values

After structural missing values have been handled, some categorical values are still missing. These remaining missing values are treated as unknown categories rather than absent features.

For this project, the remaining categorical missing values are filled using the mode from the training set. The mode is the most frequent observed category. This is a simple and transparent imputation method, and it avoids introducing additional modelling assumptions into the cleaning stage.

A more advanced approach could impute missing categories using related variables, such as garage area, basement area, year built, or overall quality. However, this would require a more detailed investigation and would introduce extra assumptions. For this project, mode imputation is used as a practical baseline.

The imputation values are calculated from the training set only, then applied to the training set, test set, and inherited houses dataset. This avoids using information from the test set when preparing the data.

In [ ]:
def get_missing_object_columns(*dfs):
    """
    Return object columns with missing values in one or more dataframes.
    """
    missing_cols = set()

    for df in dfs:
        object_cols = df.select_dtypes(include=["object"]).columns
        cols_with_missing = object_cols[df[object_cols].isna().sum() > 0]
        missing_cols.update(cols_with_missing)

    return sorted(missing_cols)

In [ ]:
remaining_categorical_missing_cols = get_missing_object_columns(
    TrainSet,
    TestSet,
    inherited_houses_df
)

remaining_categorical_missing_cols

In [ ]:
train_category_modes = {}

for col in remaining_categorical_missing_cols:
    mode_value = TrainSet[col].mode(dropna=True)[0]
    train_category_modes[col] = mode_value

    TrainSet[col] = TrainSet[col].fillna(mode_value)
    TestSet[col] = TestSet[col].fillna(mode_value)

    if col in inherited_houses_df.columns:
        inherited_houses_df[col] = inherited_houses_df[col].fillna(mode_value)

train_category_modes

# Impute Remaining Numerical Missing Values

In [ ]:
def get_missing_numeric_columns(*dfs, target=None):
    """
    Return numeric columns with missing values in one or more dataframes.
    """
    missing_cols = set()

    for df in dfs:
        numeric_cols = df.select_dtypes(include=["number"]).columns
        cols_with_missing = numeric_cols[df[numeric_cols].isna().sum() > 0]
        missing_cols.update(cols_with_missing)

    if target is not None and target in missing_cols:
        missing_cols.remove(target)

    return sorted(missing_cols)

In [ ]:
remaining_numeric_missing_cols = get_missing_numeric_columns(
    TrainSet,
    TestSet,
    inherited_houses_df,
    target="SalePrice"
)

remaining_numeric_missing_cols

In [ ]:
train_medians = TrainSet[remaining_numeric_missing_cols].median()

for col in remaining_numeric_missing_cols:
    TrainSet[col] = TrainSet[col].fillna(train_medians[col])
    TestSet[col] = TestSet[col].fillna(train_medians[col])

    if col in inherited_houses_df.columns:
        inherited_houses_df[col] = inherited_houses_df[col].fillna(train_medians[col])

train_medians

For several numerical columns, the training-set median is `0`. This is meaningful for area-based features such as `2ndFlrSF`, `EnclosedPorch`, and `WoodDeckSF`, where `0` represents the absence of that feature. For `LotFrontage` and `BedroomAbvGr`, the median provides a simple central-value imputation.

# Validate Cleaned Data

The following checks confirm that no missing values remain after the cleaning process.

In [ ]:
missing_values_table(TrainSet)

In [ ]:
missing_values_table(TestSet)

In [ ]:
missing_values_table(inherited_houses_df)

In [ ]:
print("TrainSet missing values:", TrainSet.isna().sum().sum())
print("TestSet missing values:", TestSet.isna().sum().sum())
print("Inherited houses missing values:", inherited_houses_df.isna().sum().sum())

Also shapes:

In [ ]:
TrainSet.shape, TestSet.shape, inherited_houses_df.shape

And columns:

In [ ]:
set(TrainSet.columns) - set(TestSet.columns), set(TestSet.columns) - set(TrainSet.columns)

In [ ]:
set([col for col in TrainSet.columns if col != "SalePrice"]) - set(inherited_houses_df.columns)

# Save Cleaned Datasets

In [ ]:
cleaned_data_dir = Path("outputs/datasets/cleaned")
cleaned_data_dir.mkdir(parents=True, exist_ok=True)

TrainSet.to_csv(cleaned_data_dir / "TrainSetCleaned.csv", index=False)
TestSet.to_csv(cleaned_data_dir / "TestSetCleaned.csv", index=False)
inherited_houses_df.to_csv(cleaned_data_dir / "InheritedHousesCleaned.csv", index=False)

# Conclusions and Next Steps

The data cleaning process checked duplicated rows and missing values in the housing and inherited houses datasets.

The main cleaning decisions were:

* duplicated rows were removed if present;
* missing categorical values were only replaced with `"None"` where other columns showed that the feature was absent;
* missing `GarageYrBlt` values were replaced with `0` only where `GarageArea` showed that the house had no garage;
* missing `MasVnrArea` values were replaced with `0`, since `0` is a valid value for no masonry veneer area;
* missing `BsmtFinType1` values were replaced with `"Unf"` where the basement existed but `BsmtFinSF1` was `0`;
* remaining categorical missing values were filled using the mode from the training set;
* remaining numerical missing values were filled using the median from the training set.

The same training-set imputation values were applied to the test set and inherited houses dataset to avoid using test-set information when preparing the data.

The cleaned train, test, and inherited houses datasets were saved to `outputs/datasets/cleaned`.

The next notebook will focus on feature engineering, including encoding categorical variables, preparing features, and selecting the inputs for the machine learning pipeline.